In [1]:
"""
SyntheticEpiDataBuilder
=======================
Generates synthetic epidemic datasets with known spatial structure,
intended for validating that GNNs with the correct graph outperform
those with wrong graphs (e.g. geographic_neighbors should beat identity
when spread is strictly neighbour-to-neighbour).

Design goals
------------
- Output a long-format pd.DataFrame with columns compatible with
  EpiDataOrchestrator: [timestamp, nuts_node, cases, population_size]
- Support pluggable spread methods via a Strategy protocol
- Be fully reproducible via a seed
- Make the spatial signal *strong enough* that a correctly structured
  GNN must detect it

Python features demonstrated
-----------------------------
- np.random.default_rng()       : modern numpy RNG (not np.random.seed)
- @dataclass + field()          : typed config containers
- Protocol                      : structural subtyping / duck-typing
- itertools.product             : cartesian product for DataFrame init
- __post_init__                 : dataclass validation hook
- abstractmethod-free strategy  : just match the Protocol signature
"""

from __future__ import annotations

import itertools
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable, Optional
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

@dataclass
class SyntheticEpiConfig:
    """
    Configuration for synthetic epidemic generation.

    Parameters
    ----------
    node_ids : list[int]
        NUTS node integer IDs — must match your tokenization order.
    timestamps : list
        Ordered list of timestamps (e.g. pd.date_range output).
    population_per_node : int | dict[int, int]
        Either a single int (same for all nodes) or a dict mapping node_id -> population.
    seed : int
        Random seed for reproducibility.
    train_frac : float
        Fraction of timestamps to mark as training data.
    val_frac : float
        Fraction of timestamps to mark as validation data.
        Remaining becomes test.
    """
    node_ids:             list
    timestamps:           list
    population_per_node:  int | dict = 10_000
    seed:                 int        = 42
    train_frac:           float      = 0.6
    val_frac:             float      = 0.2

    def __post_init__(self):
        # Validate fracs sum to <= 1
        if self.train_frac + self.val_frac >= 1.0:
            raise ValueError(
                f"train_frac ({self.train_frac}) + val_frac ({self.val_frac}) must be < 1.0"
            )
        if len(self.node_ids) == 0:
            raise ValueError("node_ids cannot be empty")
        if len(self.timestamps) == 0:
            raise ValueError("timestamps cannot be empty")

    def population(self, node_id: int) -> int:
        """Returns population for a given node."""
        if isinstance(self.population_per_node, dict):
            return self.population_per_node[node_id]
        return self.population_per_node


# ---------------------------------------------------------------------------
# Spread Strategy Protocol
# ---------------------------------------------------------------------------

@runtime_checkable
class SpreadStrategy(Protocol):
    """
    Duck-typed interface for epidemic spread strategies.

    Any class or callable that matches this signature is a valid strategy.
    No inheritance required — this is structural subtyping via Protocol.

    Parameters
    ----------
    config      : SyntheticEpiConfig
    adjacency   : dict[int, list[int]]
        Maps node_id -> list of neighbour node_ids (from your graph).
    rng         : np.random.Generator
        Pass the shared RNG so simulations are fully reproducible.

    Returns
    -------
    pd.DataFrame with columns: [timestamp, nuts_node, cases]
        One row per (timestamp, node) pair.
    """
    def simulate(
        self,
        config:     SyntheticEpiConfig,
        adjacency:  dict[int, list[int]],
        rng:        np.random.Generator,
    ) -> pd.DataFrame:
        ...


# ---------------------------------------------------------------------------
# Concrete Strategies
# ---------------------------------------------------------------------------

@dataclass
class NeighborSIRSpread:
    """
    Strict geographic neighbor SIR spread.

    This is the key strategy for testing whether geographic_neighbors
    outperforms identity_graph: infection only moves along edges in the
    adjacency dict. A model without those edges *cannot* predict spread.

    Parameters
    ----------
    beta : float
        Per-contact infection probability per timestep.
    gamma : float
        Recovery rate per timestep.
    initial_infected : int
        Number of initially infected nodes (chosen at random).
    noise_scale : float
        Gaussian noise added to case counts (simulates reporting noise).
        Set to 0.0 for a perfectly clean signal.
    """
    beta:              float = 0.3
    gamma:             float = 0.05
    initial_infected:  int   = 1
    noise_scale:       float = 0.05   # fraction of population

    def simulate(
        self,
        config:    SyntheticEpiConfig,
        adjacency: dict[int, list[int]],
        rng:       np.random.Generator,
    ) -> pd.DataFrame:

        nodes     = config.node_ids
        n_nodes   = len(nodes)
        n_times   = len(config.timestamps)
        node_idx  = {n: i for i, n in enumerate(nodes)}

        # --- Initialise SIR compartments (fractions of population) ---
        S = np.ones(n_nodes)
        I = np.zeros(n_nodes)
        R = np.zeros(n_nodes)

        # Seed initial outbreak at random nodes
        outbreak_nodes = rng.choice(n_nodes, size=self.initial_infected, replace=False)
        for idx in outbreak_nodes:
            I[idx] = 0.01   # 1% initially infected
            S[idx] = 0.99

        records = []

        for t, ts in enumerate(config.timestamps):
            # Record cases at this timestep
            for node_id in nodes:
                i = node_idx[node_id]
                pop = config.population(node_id)
                raw_cases = I[i] * pop
                # Add observational noise
                if self.noise_scale > 0:
                    noise = rng.normal(0, self.noise_scale * pop)
                    raw_cases = max(0.0, raw_cases + noise)
                records.append({
                    'timestamp':  ts,
                    'nuts_node':  node_id,
                    'cases':      int(round(raw_cases)),
                })

            # --- SIR update (only spread to adjacency neighbours) ---
            dS = np.zeros(n_nodes)
            dI = np.zeros(n_nodes)
            dR = np.zeros(n_nodes)

            for node_id in nodes:
                i         = node_idx[node_id]
                neighbours= adjacency.get(node_id, [])

                # Force of infection: sum of infected neighbours
                # (identity graph has no neighbours => no spread => flat signal)
                neighbor_infected = sum(I[node_idx[nb]] for nb in neighbours if nb in node_idx)
                n_neighbors       = max(len(neighbours), 1)  # avoid division by zero

                lambda_i  = self.beta * neighbor_infected / n_neighbors
                new_inf   = lambda_i * S[i]
                new_rec   = self.gamma * I[i]

                dS[i] -= new_inf
                dI[i] += new_inf - new_rec
                dR[i] += new_rec

            S = np.clip(S + dS, 0, 1)
            I = np.clip(I + dI, 0, 1)
            R = np.clip(R + dR, 0, 1)

        return pd.DataFrame(records)


@dataclass
class IndependentNoisySpread:
    """
    Each node evolves independently — no spatial coupling at all.
    An identity_graph should match or beat geographic_neighbors here,
    since there's nothing spatial to learn.

    Useful as a negative control: if geographic_neighbors still wins
    on this, something is wrong with your evaluation.
    """
    ar_coef:     float = 0.7   # autoregressive coefficient
    noise_scale: float = 0.1

    def simulate(
        self,
        config:    SyntheticEpiConfig,
        adjacency: dict[int, list[int]],   # intentionally ignored
        rng:       np.random.Generator,
    ) -> pd.DataFrame:

        nodes    = config.node_ids
        n_nodes  = len(nodes)
        n_times  = len(config.timestamps)

        # AR(1) process per node, fully independent
        series = np.zeros((n_times, n_nodes))
        series[0] = rng.uniform(0, 0.05, size=n_nodes)

        for t in range(1, n_times):
            noise        = rng.normal(0, self.noise_scale, size=n_nodes)
            series[t]    = np.clip(self.ar_coef * series[t - 1] + noise, 0, None)

        records = []
        for t, ts in enumerate(config.timestamps):
            for j, node_id in enumerate(nodes):
                pop = config.population(node_id)
                records.append({
                    'timestamp': ts,
                    'nuts_node': node_id,
                    'cases':     int(round(series[t, j] * pop)),
                })

        return pd.DataFrame(records)


# ---------------------------------------------------------------------------
# Main Builder
# ---------------------------------------------------------------------------

class SyntheticEpiDataBuilder:
    """
    Builds a synthetic epidemic dataset from a SpreadStrategy and a graph
    adjacency structure.

    Usage
    -----
    import pandas as pd
    from src.graphconstruction import StaticGraphOrchestrator

    # 1. Extract adjacency from your graph registry
    graph_entry = static_graphconstruction_nuts3.graph_registry['geographical_neighbors1']
    adjacency   = SyntheticEpiDataBuilder.adjacency_from_graphentry(graph_entry, node_ids)

    # 2. Configure
    timestamps = pd.date_range('2018-01-01', periods=104, freq='W-MON')
    config = SyntheticEpiConfig(
        node_ids   = node_ids,
        timestamps = list(timestamps),
        seed       = 42,
    )

    # 3. Build
    builder = SyntheticEpiDataBuilder(config, strategy=NeighborSIRSpread(beta=0.3))
    df      = builder.build(adjacency)

    # 4. Feed into EpiDataOrchestrator via from_raw() or a custom reader
    """

    def __init__(
        self,
        config:   SyntheticEpiConfig,
        strategy: SpreadStrategy,
    ):
        # Runtime Protocol check — catches wrong objects early with a clear message
        if not isinstance(strategy, SpreadStrategy):
            raise TypeError(
                f"strategy must implement SpreadStrategy.simulate(), got {type(strategy)}"
            )
        self.config   = config
        self.strategy = strategy
        # Modern numpy RNG — prefer this over np.random.seed() globally
        self._rng     = np.random.default_rng(config.seed)

    def build(self, adjacency: dict[int, list[int]]) -> pd.DataFrame:
        """
        Run the spread simulation and return a DataFrame ready for
        EpiDataOrchestrator ingestion.

        Columns
        -------
        timestamp, nuts_node, cases, population_size, train, val, test
        """
        # Simulate spread
        sim_df = self.strategy.simulate(self.config, adjacency, self._rng)

        # Add population
        sim_df['population_size'] = sim_df['nuts_node'].map(
            lambda nid: self.config.population(nid)
        )

        # Add temporal splits
        sim_df = self._add_splits(sim_df)

        # Sort for readability
        sim_df = sim_df.sort_values(['timestamp', 'nuts_node']).reset_index(drop=True)

        return sim_df

    def _add_splits(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Adds boolean train/val/test columns based on timestamp order.
        Mimics EpiNormalizer._set_splits logic.
        """
        timestamps = sorted(df['timestamp'].unique())
        n          = len(timestamps)

        train_end  = timestamps[int(n * self.config.train_frac)]
        val_end    = timestamps[int(n * (self.config.train_frac + self.config.val_frac))]

        df = df.copy()
        df['train'] = df['timestamp'] < train_end
        df['val']   = (df['timestamp'] >= train_end) & (df['timestamp'] < val_end)
        df['test']  = df['timestamp'] >= val_end

        return df

    # ------------------------------------------------------------------
    # Utility: extract adjacency from your existing graph registry
    # ------------------------------------------------------------------

    @staticmethod
    def adjacency_from_graphentry(
        graph_entry,            # GraphEntry from your registry
        node_ids: list[int],    # ordered list of integer node IDs
        threshold: float = 0.0, # minimum edge weight to include
    ) -> dict[int, list[int]]:
        """
        Converts a GraphEntry (edge_index + edge_weight tensors) into a
        plain Python adjacency dict: {node_id: [neighbour_node_id, ...]}.

        Parameters
        ----------
        graph_entry : GraphEntry
            From static_graphconstruction_nuts3.graph_registry[name]
        node_ids : list[int]
            Must be in the same integer-index order as your tokenization map.
        threshold : float
            Edges with weight <= threshold are dropped (useful for eps self-loops).

        Returns
        -------
        dict mapping each node_id to its list of neighbours.
        """
        edge_index  = graph_entry.graphstructure.edge_index   # [2, E]
        edge_weight = graph_entry.graphstructure.edge_weight  # [E]

        # Build adjacency using torch boolean masking — more readable than looping
        mask        = edge_weight > threshold
        src_nodes   = edge_index[0][mask].tolist()
        dst_nodes   = edge_index[1][mask].tolist()

        adjacency: dict[int, list[int]] = {nid: [] for nid in node_ids}

        for src_idx, dst_idx in zip(src_nodes, dst_nodes):
            # src_idx and dst_idx are token indices, not node_ids
            if src_idx < len(node_ids) and dst_idx < len(node_ids):
                src_id = node_ids[src_idx]
                dst_id = node_ids[dst_idx]
                if src_id != dst_id:  # exclude self-loops
                    adjacency[src_id].append(dst_id)

        return adjacency

    def summary(self, df: pd.DataFrame) -> None:
        """
        Print a quick sanity-check summary of the synthetic dataset.
        Useful for verifying the spatial signal before training.
        """
        n_nodes = df['nuts_node'].nunique()
        n_times = df['timestamp'].nunique()
        total   = df['cases'].sum()
        peak    = df.groupby('timestamp')['cases'].sum().idxmax()

        # groupby + agg in one shot — a cleaner pattern than multiple .mean() calls
        node_stats = (
            df.groupby('nuts_node')['cases']
            .agg(mean='mean', std='std', max='max')
        )
        spatial_cv = (node_stats['std'] / (node_stats['mean'] + 1e-6)).mean()

        print(f"SyntheticEpiData Summary")
        print(f"  Nodes:        {n_nodes}")
        print(f"  Timesteps:    {n_times}")
        print(f"  Total cases:  {total:,}")
        print(f"  Peak week:    {peak}")
        print(f"  Spatial CV:   {spatial_cv:.3f}  (higher = more spatial variation to exploit)")
        print(f"  Train steps:  {df['train'].any() and df[df['train']]['timestamp'].nunique()}")
        print(f"  Val steps:    {df[df['val']]['timestamp'].nunique()}")
        print(f"  Test steps:   {df[df['test']]['timestamp'].nunique()}")


# ---------------------------------------------------------------------------
# Example usage (run as script to verify)
# ---------------------------------------------------------------------------


In [ ]:
import pandas as pd

# Tiny toy graph: 6 nodes in a chain: 0-1-2-3-4-5
node_ids   = list(range(6))
timestamps = list(pd.date_range('2020-01-06', periods=52, freq='W-MON'))

config = SyntheticEpiConfig(
    node_ids            = node_ids,
    timestamps          = timestamps,
    population_per_node = 50_000,
    seed                = 42,
)

# Chain adjacency: each node knows only its immediate neighbours
chain_adjacency = {
    0: [1],
    1: [0, 2],
    2: [1, 3],
    3: [2, 4],
    4: [3, 5],
    5: [4],
}

# Identity adjacency: no neighbours (simulates identity graph)
identity_adjacency = {i: [] for i in node_ids}

# Build with neighbor spread strategy
builder = SyntheticEpiDataBuilder(
    config   = config,
    strategy = NeighborSIRSpread(beta=0.4, gamma=0.05, initial_infected=1, noise_scale=0.02),
)

df_chain    = builder.build(chain_adjacency)
builder._rng = np.random.default_rng(config.seed)   # reset rng for fair comparison
df_identity = builder.build(identity_adjacency)

print("=== Chain adjacency (geographic_neighbors should exploit this) ===")
builder.summary(df_chain)

print("\n=== Identity adjacency (flat — identity graph does fine here) ===")
builder.summary(df_identity)

print("\nFirst 12 rows of chain dataset:")
print(df_chain.head(12).to_string(index=False))